In [57]:
from main import load_model_from_study
import spacy

import optuna

import torch.nn as nn
import torch

from torch.utils.data import DataLoader
from src.model import evaluate_model
from src.dataset import YelpDataset

from sklearn.metrics import accuracy_score

In [58]:
import numpy as np
import pandas as pd

In [59]:
NLP = spacy.load('en_core_web_lg')
NLP_DISABLE = ['ner', 'parser', 'attribute_ruler', 'lemmatizer', 'tagger']
MAX_SEQ_LEN = 100
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [60]:
# Optuna variables
study_name="sentiment_lstm_hpo"
direction="minimize"
storage="sqlite:///./models/sentiment_lstm_hpo.db"

# Load the optuna study
study = optuna.load_study(study_name=study_name, storage=storage)

In [61]:
model = load_model_from_study(study)
loss_fn = nn.CrossEntropyLoss()

In [62]:
df = pd.read_csv('./data/yelp_review_100k.csv', skiprows=range(1, READ_SIZE+1))
df['length'] = df['text'].apply(lambda x: len(x.split()))
df['y'] = np.array(df['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2}))

In [63]:
df_grouped = df.groupby('length')

In [64]:
df_seq_acc = pd.DataFrame(columns=['seq_len', 'accuracy'])

In [66]:
for length, group in df_grouped:

    print (f'len {length}')


    tokenized_vectors = []
    
    # Tokenize sentences.
    for doc in NLP.pipe(group['text'], 
                        batch_size=10, 
                        disable=NLP_DISABLE, 
                        n_process=1
                        ):
        
        # Record the tokens that have a vector.
        sentence = []
        for token in doc:
            if token.has_vector:
                sentence.append(token.vector)

        # Append the vectorized sample.
        tokenized_vectors.append(np.array(sentence))

    group['embeddings'] = tokenized_vectors
    dataset = YelpDataset(group['embeddings'], group['y'], MAX_SEQ_LEN)
    loader =  DataLoader(dataset, batch_size=len(group['text']))

    group['y_pred'], _, _ = evaluate_model(model, loader, loss_fn, DEVICE)

    acc = accuracy_score(group['y'], group['y_pred'])
    print (f'seq_len {length} acc {acc}')

    # Append the new sample using loc (recommended for adding rows one at a time)
    df_seq_acc.loc[len(df_seq_acc)] = [length, acc]

len 1


Evaluating: 100%|██████████| 1/1 [00:00<00:00,  5.26it/s]


seq_len 1 acc 0.0
len 2


Evaluating: 100%|██████████| 1/1 [00:00<00:00, 428.12it/s]


seq_len 2 acc 1.0
len 3


...

seq_len 951 acc 1.0
len 959


Evaluating: 100%|██████████| 1/1 [00:00<?, ?it/s]

seq_len 959 acc 1.0
len 974



Evaluating: 100%|██████████| 1/1 [00:00<?, ?it/s]

seq_len 974 acc 0.0
